# Run `amir/score.py` in Colab

Open this notebook in Google Colab, then choose `Runtime -> Change runtime type` and select a GPU runtime.

This workflow is wired for the `amir/transcripts` corpus. GPU is supported through PyTorch + Transformers. TPU runtimes can still open the notebook, but `score.py` currently falls back to CPU unless you add `torch-xla` support.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Ngafney/garda-spring26.git"
REPO_DIR = Path("/content/garda-spring26") if IN_COLAB else Path.cwd()
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_REPO_DIR = Path("/content/drive/MyDrive/garda-spring26")

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = GOOGLE_DRIVE_REPO_DIR

if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {REPO_DIR}")

Working directory: /content/garda-spring26


In [2]:
%pip install -q -r requirements.txt tomli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/131.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 114.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.

In [3]:
import os
import sys
import torch

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
if os.environ.get("COLAB_TPU_ADDR"):
    print("TPU runtime detected. score.py will still use CPU unless torch-xla support is added.")

Python: 3.12.13
CUDA available: True
CUDA device count: 1
GPU: Tesla T4


## Parameters

Set `LIMIT = None` for the full corpus. There are about 9,951 transcript files in `amir/transcripts`, so a full run can take a long time even on Colab GPU.

Use `PATTERN` to score a subset first, for example `"amazon"` or `"nvidia"`.

If you already have `amir/transcripts_metadata.csv`, leave `REBUILD_METADATA = False` and the notebook will use that file.

In [4]:
TRANSCRIPTS_DIR = REPO_DIR / "amir" / "transcripts"
OUTPUT_DIR = REPO_DIR / "amir" / "outputs"
METADATA_CSV = REPO_DIR / "amir" / "transcripts_metadata.csv"
OUTPUT_CSV = OUTPUT_DIR / "transcript_scores.csv"
SCORE_DB = OUTPUT_DIR / "score_cache.sqlite"

LIMIT = 25
PATTERN = None
FORCE = True
REBUILD_METADATA = False
VERBOSE = 1

print(TRANSCRIPTS_DIR)
print(METADATA_CSV)
print(OUTPUT_CSV)

/content/garda-spring26/amir/transcripts
/content/garda-spring26/amir/outputs/transcript_scores.csv


In [6]:
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    "amir/score.py",
    "--transcripts-dir",
    str(TRANSCRIPTS_DIR),
    "--metadata-csv",
    str(METADATA_CSV),
    "--output-csv",
    str(OUTPUT_CSV),
    "--score-db",
    str(SCORE_DB),
]

if LIMIT is not None:
    cmd += ["--limit", str(LIMIT)]
if PATTERN:
    cmd += ["--pattern", PATTERN]
if REBUILD_METADATA:
    cmd.append("--rebuild-metadata")
if FORCE:
    cmd.append("--force")
cmd += ["-v"] * VERBOSE

print("Running:")
print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)

Running:
/usr/bin/python3 amir/score.py --transcripts-dir /content/garda-spring26/amir/transcripts --metadata-csv /content/garda-spring26/amir/outputs/transcript_metadata.csv --output-csv /content/garda-spring26/amir/outputs/transcript_scores.csv --score-db /content/garda-spring26/amir/outputs/score_cache.sqlite --rebuild-metadata --limit 25 --force -v


CalledProcessError: Command '['/usr/bin/python3', 'amir/score.py', '--transcripts-dir', '/content/garda-spring26/amir/transcripts', '--metadata-csv', '/content/garda-spring26/amir/outputs/transcript_metadata.csv', '--output-csv', '/content/garda-spring26/amir/outputs/transcript_scores.csv', '--score-db', '/content/garda-spring26/amir/outputs/score_cache.sqlite', '--rebuild-metadata', '--limit', '25', '--force', '-v']' returned non-zero exit status 1.

In [ ]:
import pandas as pd

scores = pd.read_csv(OUTPUT_CSV)
print(f"Rows written: {len(scores):,}")
display(scores.head())